# Ingestión del archivo `movie_cast.json`

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

## 1. Leer el archivo JSON usando `DataFrameReader` de Spark

In [0]:
movie_cast_schema = "movieId INT, personId INT, characterName STRING, genderId INT, castOrder INT"

movie_cast_df = (spark.read 
    .schema(movie_cast_schema)
    .option("multiLine", True)
    .json(f"{bronze_folder_path}/{v_file_date}/movie_cast.json")
)
display(movie_cast_df)

movieId,personId,characterName,genderId,castOrder
285,85,Captain Jack Sparrow,2,0
285,114,Will Turner,2,1
285,116,Elizabeth Swann,1,2
285,1640,William Bootstrap Bill Turner,2,3
285,1619,Captain Sao Feng,2,4
285,2440,Captain Davy Jones,2,5
285,118,Captain Hector Barbossa,2,6
285,1709,Admiral James Norrington,2,7
285,2449,Joshamee Gibbs,2,8
285,2441,Lord Cutler Beckett,2,9


## 2. Eliminar las columnas no deseadas del DataFrame

In [0]:
movie_cast_dropped_df = movie_cast_df.drop("genderId", "castOrder")

## 3. Cambiar el nombre de las columnas según lo requerido

In [0]:
movie_cast_renamed_df = (movie_cast_dropped_df
    .withColumnRenamed("movieId", "movie_id")
    .withColumnRenamed("personId", "person_id")
    .withColumnRenamed("characterName", "character_name")
)

## 4. Agregar las columnas `ingestion_date` y `environmate` al DateFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

movie_cast_final_df = add_ingestion_date(movie_cast_renamed_df).withColumn("enviroment", lit(v_environment)).withColumn("file_date", lit(v_file_date))


## 5. Escribir datos en el datalake en formato `Parquet`

In [0]:
merge_delta_lake( movie_cast_final_df, "movie_silver", "movies_casts", "tgt.movie_id = src.movie_id AND tgt.file_date = src.file_date AND tgt.person_id = src.person_id", "file_date" )

In [0]:
%sql
SELECT * FROM movie_silver.movies_casts

path,name,size,modificationTime
abfss://silver@moviehistory4.dfs.core.windows.net/movie_casts/_SUCCESS,_SUCCESS,0,1789060840000
abfss://silver@moviehistory4.dfs.core.windows.net/movie_casts/_committed_719293224871520935,_committed_719293224871520935,233,1789060840000
abfss://silver@moviehistory4.dfs.core.windows.net/movie_casts/_committed_7601461840224044453,_committed_7601461840224044453,124,1789060708000
abfss://silver@moviehistory4.dfs.core.windows.net/movie_casts/_started_719293224871520935,_started_719293224871520935,0,1789060838000
abfss://silver@moviehistory4.dfs.core.windows.net/movie_casts/_started_7601461840224044453,_started_7601461840224044453,0,1789060706000
abfss://silver@moviehistory4.dfs.core.windows.net/movie_casts/part-00000-tid-719293224871520935-2ce51148-99cb-4c8c-a52d-58a1fa74fab6-224-1-c000.snappy.parquet,part-00000-tid-719293224871520935-2ce51148-99cb-4c8c-a52d-58a1fa74fab6-224-1-c000.snappy.parquet,771662,1789060839000
